# PDEForge Quickstart

This notebook demonstrates the basic usage of PDEForge for generating PDE datasets for operator learning.

## Key Features

1. **Unified API**: Same interface for all PDE models
2. **Configurable**: Resolution, domain, parameters all adjustable
3. **Interactive**: Built-in visualization widgets
4. **Extensible**: Easy to add new models

In [ ]:
# Install PDEForge (if not already installed)
# !pip install -e ..

import numpy as np
import matplotlib.pyplot as plt

# Import PDEForge
import sys
sys.path.insert(0, '..')

from pdeforge import generate_dataset, list_models, get_model

## 1. List Available Models

PDEForge provides several built-in PDE models:

In [ ]:
print("Available models:")
for model in list_models():
    print(f"  - {model}")

## 2. Generate a 1D Burgers Dataset

The 1D Burgers equation models advection-diffusion with shock formation:

$$\frac{\partial u}{\partial t} + \mu u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

**Task**: Learn the mapping $u(x, t=0) \rightarrow u(x, t=T)$

In [ ]:
# Generate dataset with unified API
burgers_dataset = generate_dataset(
    model="burgers_1d",
    n_samples=100,
    resolution={"x": 256},
    params={
        "viscosity": 0.01,
        "advection": 1.0,
        "time_end": 1.0,
    },
    seed=42,
)

print(burgers_dataset)

In [ ]:
# Visualize a few samples
x = burgers_dataset.grid['x']

fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for i in range(3):
    # Input (initial condition)
    axes[0, i].plot(x, burgers_dataset.inputs[i], 'b-', linewidth=2)
    axes[0, i].set_title(f'Sample {i+1}: u(x, t=0)')
    axes[0, i].set_xlabel('x')
    axes[0, i].grid(True, alpha=0.3)
    
    # Output (solution at t=T)
    axes[1, i].plot(x, burgers_dataset.outputs[i], 'r-', linewidth=2)
    axes[1, i].set_title(f'Sample {i+1}: u(x, t=T)')
    axes[1, i].set_xlabel('x')
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Generate a 2D Stokes Flow Dataset

The Stokes equations describe creeping viscous flow:

$$-\mu \nabla^2 \mathbf{u} + \nabla p = \mathbf{f}, \quad \nabla \cdot \mathbf{u} = 0$$

**Task**: Learn the mapping $(f_x, f_y) \rightarrow (u, v, p)$

In [ ]:
# Generate Stokes dataset
stokes_dataset = generate_dataset(
    model="stokes_2d",
    n_samples=50,
    resolution={"x": 64, "y": 64},
    params={
        "viscosity": 1.0,
        "n_force_modes": 5,
    },
    seed=42,
)

print(stokes_dataset)

In [ ]:
# Visualize a sample
idx = 0
x = stokes_dataset.grid['x']
y = stokes_dataset.grid['y']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Inputs: force components
im0 = axes[0, 0].contourf(x, y, stokes_dataset.inputs[idx, :, :, 0], levels=20, cmap='RdBu_r')
axes[0, 0].set_title('Input: $f_x$')
axes[0, 0].set_aspect('equal')
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].contourf(x, y, stokes_dataset.inputs[idx, :, :, 1], levels=20, cmap='RdBu_r')
axes[0, 1].set_title('Input: $f_y$')
axes[0, 1].set_aspect('equal')
plt.colorbar(im1, ax=axes[0, 1])

# Force magnitude
f_mag = np.sqrt(stokes_dataset.inputs[idx, :, :, 0]**2 + stokes_dataset.inputs[idx, :, :, 1]**2)
im2 = axes[0, 2].contourf(x, y, f_mag, levels=20, cmap='viridis')
axes[0, 2].set_title('$|\mathbf{f}|$')
axes[0, 2].set_aspect('equal')
plt.colorbar(im2, ax=axes[0, 2])

# Outputs: velocity and pressure
im3 = axes[1, 0].contourf(x, y, stokes_dataset.outputs[idx, :, :, 0], levels=20, cmap='RdBu_r')
axes[1, 0].set_title('Output: $u$')
axes[1, 0].set_aspect('equal')
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].contourf(x, y, stokes_dataset.outputs[idx, :, :, 1], levels=20, cmap='RdBu_r')
axes[1, 1].set_title('Output: $v$')
axes[1, 1].set_aspect('equal')
plt.colorbar(im4, ax=axes[1, 1])

im5 = axes[1, 2].contourf(x, y, stokes_dataset.outputs[idx, :, :, 2], levels=20, cmap='viridis')
axes[1, 2].set_title('Output: $p$')
axes[1, 2].set_aspect('equal')
plt.colorbar(im5, ax=axes[1, 2])

plt.tight_layout()
plt.show()

## 4. Split Dataset for Training

PDEForge provides convenient splitting for machine learning workflows:

In [ ]:
# Split into train/val/calibration/test
splits = burgers_dataset.split(
    train=0.6,
    val=0.15,
    cal=0.15,
    test=0.1,
    seed=42,
)

for name, ds in splits.items():
    print(f"{name}: {ds.n_samples} samples")

## 5. Save and Load Datasets

Datasets can be saved in multiple formats:

In [ ]:
# Save to directory
burgers_dataset.save("./burgers_data")

# Load it back
from pdeforge.io import load_dataset
loaded = load_dataset("./burgers_data")
print(f"Loaded: {loaded}")

In [ ]:
# Clean up
import shutil
shutil.rmtree("./burgers_data")

## 6. Direct Model Access

For more control, you can access models directly:

In [ ]:
# Get model class from registry
BurgersModel = get_model("burgers_1d")

# Create instance with custom parameters
model = BurgersModel(
    resolution={"x": 128},
    viscosity=0.001,  # Lower viscosity = sharper shocks
)

# Generate a single sample
ic, solution, info = model.generate_sample(seed=123)

print(f"Initial condition shape: {ic.shape}")
print(f"Solution shape: {solution.shape}")
print(f"Valid: {info['valid']}")

In [ ]:
# Plot the sample
x = model.grids['x']

plt.figure(figsize=(10, 4))
plt.plot(x, ic, 'b-', label='u(x, t=0)', linewidth=2)
plt.plot(x, solution, 'r-', label='u(x, t=T)', linewidth=2)
plt.xlabel('x')
plt.ylabel('u')
plt.title('Burgers Equation: Low Viscosity Shock Formation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

PDEForge provides:

1. **Unified API** via `generate_dataset()` - same interface for all models
2. **Built-in models**: Burgers 1D, Darcy 2D, Stokes 2D, Cylinder Flow 2D (FEniCSx)
3. **Flexible configuration**: resolution, domain, all physical parameters
4. **Data management**: split, save, load in multiple formats
5. **Extensibility**: easy to add new models via registry pattern

### Other Notebooks

**Visualization:**
- `05_interactive_visualization.ipynb` - Interactive visualization (steady cylinder flow + Burgers 1D)
- `06_unsteady_cylinder_flow.ipynb` - Time-dependent vortex shedding with animation

**For UQ workflows:**
- `03_uq_workflow.ipynb` - Using PDEForge with Operator_UQ for uncertainty quantification

**For contributors:**
- `02_adding_fenicsx_models.ipynb` - How to add FEniCSx-based models

**Coming soon:**
- `04_stochastic_models.ipynb` - Generating stochastic PDE datasets